# Your first ML project, step by step
Run each cell against the original Palmer Penguins CSV.


**[Watch the video](https://youtu.be/Cy0eILNBKm8)** 


## Set up the project files

If you opened this notebook in Google Colab, the next cell downloads the same project script and original CSV shipped beside this notebook in the repository. If you cloned the repository, it uses the local copies. Run the cells in order.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

PILOT = Path.cwd()
BASE = "https://raw.githubusercontent.com/G0rav/machine_learning_explained_visually_free/main/07-first-ml-project/"
for relative in ("project.py", "data/penguins.csv", "data/SOURCE.md"):
    destination = PILOT / relative
    if not destination.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(BASE + relative, destination)

csv_hash = hashlib.sha256((PILOT / "data/penguins.csv").read_bytes()).hexdigest()
assert csv_hash == "f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93", "The CSV differs from the video dataset"
print("Project files ready")

Project files ready


## Get the project files

From the project folder: `pip install -r requirements.txt`, then `python project.py`.

In [2]:
from pathlib import Path
PILOT = Path.cwd()
print('project.py:', (PILOT / 'project.py').is_file(),
      'CSV:', (PILOT / 'data/penguins.csv').is_file())

project.py: True CSV: True


## Read the original CSV

In [3]:
import csv
import numpy as np
DATA = PILOT / 'data/penguins.csv'
with DATA.open(newline='', encoding='utf-8') as handle:
    source = list(csv.DictReader(handle))
print('raw rows:', len(source), 'first species:', source[0]['species'])

raw rows: 344 first species: Adelie


## Choose two features and one label

In [4]:
FEATURES = ('bill_length_mm', 'bill_depth_mm')
clean, labels, removed = [], [], []
for source_index, row in enumerate(source):
    raw = [row[name] for name in FEATURES]
    if any(value in ('', 'NA') for value in raw):
        removed.append(source_index)
        continue
    clean.append([float(value) for value in raw])
    labels.append(row['species'])
x = np.asarray(clean, dtype=float)
y = np.asarray(labels)
print('X:', x.shape, 'y:', y.shape, 'first row:', x[0], y[0])

X: (342, 2) y: (342,) first row: [39.1 18.7] Adelie


## Account for incomplete rows

In [5]:
# The selected measurements were checked in the previous cell.
missing_source_rows = [index + 1 for index in removed]
print('removed source rows:', missing_source_rows, 'usable:', len(x))

removed source rows: [4, 272] usable: 342


## Reserve the test birds

In [6]:
from sklearn.model_selection import train_test_split
SEED = 20260925
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.20,
    random_state=SEED, stratify=y
)
x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print('train:', len(train_idx), 'sealed test:', len(test_idx))

train: 273 sealed test: 69


## Build a training-only baseline

In [7]:
from collections import Counter
CLASSES = ('Adelie', 'Chinstrap', 'Gentoo')
train_counts = Counter(y_train)
majority = max(CLASSES, key=lambda name: train_counts[name])
baseline_predictions = np.repeat(majority, len(test_idx))
print({str(name): train_counts[name] for name in CLASSES},
      'baseline:', majority,
      'predictions:', len(baseline_predictions))

{'Adelie': 121, 'Chinstrap': 54, 'Gentoo': 98} baseline: Adelie predictions: 69


## Learn the scale from training rows

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(x_train)
example = np.array([[49.8, 16.8]])
standardized = scaler.transform(example)
print('training mean:', scaler.mean_.round(2),
      'standardized:', standardized.round(2))

training mean: [43.82 17.12] standardized: [[ 1.15 -0.16]]


## Keep the steps together

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED),
)
print('pipeline steps:', [name for name, _ in model.steps])

pipeline steps: ['standardscaler', 'logisticregression']


## Fit on training birds only

In [10]:
model.fit(x_train, y_train)
print('fit rows:', len(x_train), 'test rows in fit:', 0)

fit rows: 273 test rows in fit: 0


## Predict the held-out birds

In [11]:
predictions = model.predict(x_test)
probabilities = model.predict_proba(x_test)
correct_positions = np.flatnonzero(predictions == y_test)
pos = int(min(correct_positions, key=lambda i: probabilities[i].max()))
classes = model.named_steps['logisticregression'].classes_
print('bird:', x_test[pos], 'prediction:', predictions[pos],
      'probabilities:',
      {str(name): round(float(prob), 3)
       for name, prob in zip(classes, probabilities[pos])})

bird: [49.8 16.8] prediction: Gentoo probabilities: {'Adelie': 0.003, 'Chinstrap': 0.469, 'Gentoo': 0.528}


## Calculate baseline accuracy

In [12]:
baseline_correct = int(np.sum(baseline_predictions == y_test))
baseline_accuracy = baseline_correct / len(y_test)
print(f'baseline: {baseline_correct}/{len(y_test)} = {baseline_accuracy:.1%}')

baseline: 30/69 = 43.5%


## Calculate model accuracy

In [13]:
model_correct = int(np.sum(predictions == y_test))
model_accuracy = model_correct / len(y_test)
print(f'model: {model_correct}/{len(y_test)} = {model_accuracy:.1%}')

model: 66/69 = 95.7%


## Inspect the three mistakes

In [14]:
errors = np.flatnonzero(predictions != y_test)
for position in errors:
    print('actual:', y_test[position],
          'predicted:', predictions[position],
          'bill:', x_test[position])
print('total errors:', len(errors))

actual: Gentoo predicted: Chinstrap bill: [52.1 17. ]
actual: Chinstrap predicted: Adelie bill: [40.9 16.6]
actual: Gentoo predicted: Chinstrap bill: [44.4 17.3]
total errors: 3


## Reproduce the complete result

In [15]:
import sys
sys.path.insert(0, str(PILOT))
from project import analyze
result = analyze()
print('raw:', result['raw_rows'],
      'usable:', result['clean_rows'],
      'test:', result['test_rows'],
      'correct:', result['model_correct'])

raw: 344 usable: 342 test: 69 correct: 66


## Self-check: reproduce the video

These checks use the calculations above and the standalone project script. They catch a changed dataset, split, baseline, model result, or prediction order.

In [16]:
checks = {
    "344 raw rows": len(source) == result["raw_rows"] == 344,
    "missing source rows 4 and 272": missing_source_rows == [4, 272],
    "342 usable rows": len(y) == result["clean_rows"] == 342,
    "273 training rows": len(x_train) == result["train_rows"] == 273,
    "69 sealed test rows": len(x_test) == result["test_rows"] == 69,
    "Adelie baseline": majority == result["baseline_label"] == "Adelie",
    "30 correct baseline predictions": baseline_correct == result["baseline_correct"] == 30,
    "66 correct model predictions": model_correct == result["model_correct"] == 66,
    "3 inspected errors": len(errors) == result["model_errors"] == 3,
    "same test answers": y_test.tolist() == result["test_actual"],
    "same test predictions": predictions.tolist() == result["test_predicted"],
    "confusion matrix counts all test birds": sum(map(sum, result["confusion_matrix"])) == 69,
}
failed = [name for name, passed in checks.items() if not passed]
assert not failed, "Checks failed: " + ", ".join(failed)
print(f"{len(checks)} checks passed")

12 checks passed
